# HGCTM Revision — Phase 1A Data and Lithology Audit

This notebook audits the frozen Stage 0 files used in the submitted HGCTM-S manuscript.

It does **not** refit any model and does **not** modify the original files.

Main goals:

1. Reconstruct the data flow from 1,483 raw deposits to the 1,335-deposit model set.
2. Resolve the 64% versus 1,235 lithology inconsistency.
3. Separate resolved, other, and unknown lithology records.
4. Summarize lithology and age status by deposit type and approximate continent.
5. Export transparent audit files for the manuscript revision and response letters.


## Recommended Kaggle setup

Create a **new notebook** for this audit. Keep the submitted notebook unchanged.

Attach the Stage 0 archive or dataset containing:

- `X_family_copper.csv`
- `X_family_primary.csv`
- `covariates.csv`
- `deposit_completeness.csv`
- `lda_sweep_results.csv`
- `macrostrat_lithology.csv`
- `mineral_to_family.csv`

The loader below searches Kaggle input folders automatically and also supports an attached ZIP archive.


In [ ]:
from pathlib import Path
import os
import glob
import zipfile
import json
import numpy as np
import pandas as pd

WORK = Path("/kaggle/working")
OUT = WORK / "phase1_audit"
OUT.mkdir(parents=True, exist_ok=True)

REQUIRED = {
    "X_family_copper.csv",
    "X_family_primary.csv",
    "covariates.csv",
    "deposit_completeness.csv",
    "lda_sweep_results.csv",
    "macrostrat_lithology.csv",
    "mineral_to_family.csv",
}

def find_stage0_dir():
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working")]

    # First look for an already extracted folder containing all required files.
    for root in search_roots:
        if not root.exists():
            continue
        for cov_path in root.rglob("covariates.csv"):
            folder = cov_path.parent
            present = {p.name for p in folder.iterdir() if p.is_file()}
            if REQUIRED.issubset(present):
                return folder

    # Otherwise find and extract a likely Stage 0 ZIP.
    zip_candidates = []
    for root in search_roots:
        if root.exists():
            zip_candidates.extend(root.rglob("*.zip"))

    for zip_path in zip_candidates:
        try:
            with zipfile.ZipFile(zip_path) as zf:
                names = {Path(n).name for n in zf.namelist()}
                if REQUIRED.issubset(names):
                    extract_dir = WORK / "hgctm_stage0_extracted"
                    extract_dir.mkdir(parents=True, exist_ok=True)
                    zf.extractall(extract_dir)
                    return extract_dir
        except zipfile.BadZipFile:
            continue

    raise FileNotFoundError(
        "Could not find the complete HGCTM Stage 0 dataset. "
        "Attach the Stage 0 ZIP or dataset to this Kaggle notebook."
    )

STAGE0 = find_stage0_dir()
print("Stage 0 folder:", STAGE0)
print("Files:")
for p in sorted(STAGE0.iterdir()):
    if p.is_file():
        print(" -", p.name)


In [ ]:
# Load the frozen Stage 0 files.

X_copper = pd.read_csv(STAGE0 / "X_family_copper.csv")
X_primary = pd.read_csv(STAGE0 / "X_family_primary.csv")
cov = pd.read_csv(STAGE0 / "covariates.csv")
completeness = pd.read_csv(STAGE0 / "deposit_completeness.csv")
macro = pd.read_csv(STAGE0 / "macrostrat_lithology.csv")
mapping = pd.read_csv(STAGE0 / "mineral_to_family.csv")
lda_sweep = pd.read_csv(STAGE0 / "lda_sweep_results.csv")

tables = {
    "X_family_copper": X_copper,
    "X_family_primary": X_primary,
    "covariates": cov,
    "deposit_completeness": completeness,
    "macrostrat_lithology": macro,
    "mineral_to_family": mapping,
    "lda_sweep_results": lda_sweep,
}

print("Loaded tables:")
for name, df in tables.items():
    print(f"{name:28s} {df.shape}")


## 1. Integrity checks and definitive data flow

The original workflow contains three relevant deposit counts:

- 1,483 deposits queried in Macrostrat.
- 1,402 deposits represented in the mineral-family matrix.
- 1,335 deposits retained after requiring at least three mapped mineral-family counts.

The code below verifies these values directly from the frozen files.


In [ ]:
def assert_unique_id(df, name):
    if "Mindat_id" not in df.columns:
        raise KeyError(f"{name} has no Mindat_id column.")
    n_dup = df["Mindat_id"].duplicated().sum()
    if n_dup:
        raise ValueError(f"{name} contains {n_dup} duplicated Mindat_id values.")

for name in ["X_family_copper", "X_family_primary", "covariates",
             "deposit_completeness", "macrostrat_lithology"]:
    assert_unique_id(tables[name], name)

family_cols = [c for c in X_copper.columns if c != "Mindat_id"]
mineral_count_per_deposit = X_copper[family_cols].sum(axis=1)
model_rule_mask = mineral_count_per_deposit >= 3

flow = pd.DataFrame({
    "stage": [
        "Macrostrat coordinate queries / raw GCDD deposits",
        "Deposits with mineral-family count records",
        "Deposits satisfying >=3 mapped family counts",
        "Deposits in final covariates/model set",
    ],
    "n_deposits": [
        macro["Mindat_id"].nunique(),
        X_copper["Mindat_id"].nunique(),
        int(model_rule_mask.sum()),
        cov["Mindat_id"].nunique(),
    ]
})

print(flow.to_string(index=False))

expected_model_ids = set(X_copper.loc[model_rule_mask, "Mindat_id"])
actual_model_ids = set(cov["Mindat_id"])

print("\nID agreement:")
print("Expected model IDs not in covariates:", len(expected_model_ids - actual_model_ids))
print("Covariate IDs not expected from count rule:", len(actual_model_ids - expected_model_ids))

if expected_model_ids != actual_model_ids:
    print("WARNING: The model-set IDs do not exactly match the >=3-count rule.")
else:
    print("PASS: The 1,335 model IDs exactly match the >=3-count rule.")

flow.to_csv(OUT / "phase1_data_flow.csv", index=False)


## 2. Define lithology status transparently

The submitted code treated the following as recognized geological classes:

- felsic
- mafic
- volcanic
- carbonate
- siliciclastic
- metamorphic
- evaporite

`other` means that Macrostrat returned text, but the rule-based dictionary did not assign it to a recognized class.

`unknown` means that no usable lithology text was returned.


In [ ]:
RESOLVED_CLASSES = {
    "felsic",
    "mafic",
    "volcanic",
    "carbonate",
    "siliciclastic",
    "metamorphic",
    "evaporite",
}

def lithology_status(value):
    value = str(value).strip().lower()
    if value in RESOLVED_CLASSES:
        return "resolved"
    if value == "other":
        return "other"
    if value == "unknown" or value == "nan" or value == "":
        return "unknown"
    return "unexpected_label"

cov["lithology_status"] = cov["litho_class"].apply(lithology_status)
macro["lithology_status"] = macro["litho_class"].apply(lithology_status)

print("Final 1,335-deposit model set:")
print(cov["lithology_status"].value_counts(dropna=False).to_string())

n_model = len(cov)
n_resolved = int((cov["lithology_status"] == "resolved").sum())
n_non_unknown = int((cov["lithology_status"] != "unknown").sum())

print(f"\nStrictly resolved: {n_resolved}/{n_model} = {100*n_resolved/n_model:.1f}%")
print(f"Non-unknown, including 'other': {n_non_unknown}/{n_model} = {100*n_non_unknown/n_model:.1f}%")

if n_resolved == 860 and n_non_unknown == 1235:
    print("\nConfirmed:")
    print("- 860 is the strict resolved-lithology count.")
    print("- 1,235 excludes only 'unknown' and still includes 'other'.")


In [ ]:
# Exact class counts for both the full Macrostrat query set and final model set.

def class_count_table(df, dataset_name):
    out = (
        df["litho_class"]
        .fillna("missing")
        .astype(str)
        .str.lower()
        .value_counts(dropna=False)
        .rename_axis("litho_class")
        .reset_index(name="count")
    )
    out["percent"] = 100 * out["count"] / len(df)
    out.insert(0, "dataset", dataset_name)
    return out

lithology_counts = pd.concat([
    class_count_table(macro, "all_1483_macrostrat_records"),
    class_count_table(cov, "final_1335_model_records"),
], ignore_index=True)

print(lithology_counts.to_string(index=False))
lithology_counts.to_csv(OUT / "phase1_lithology_class_counts.csv", index=False)


## 3. Diagnose `unknown` and `other` records

This section determines whether unknown records arose from technical API failures or from successful queries with no usable lithology.

It also lists the most frequent raw descriptions behind the `other` category.


In [ ]:
print("API status for all Macrostrat records:")
print(macro["api_status"].fillna("missing").value_counts(dropna=False).to_string())

print("\nAPI status among unknown records:")
print(
    macro.loc[macro["lithology_status"] == "unknown", "api_status"]
    .fillna("missing")
    .value_counts(dropna=False)
    .to_string()
)

unknown_audit = macro.loc[
    macro["lithology_status"] == "unknown",
    ["Mindat_id", "latitude", "longitude", "litho_raw", "litho_class", "api_status"]
].copy()

other_audit = macro.loc[
    macro["lithology_status"] == "other",
    ["Mindat_id", "latitude", "longitude", "litho_raw", "litho_class", "api_status"]
].copy()

other_terms = (
    other_audit["litho_raw"]
    .fillna("")
    .value_counts(dropna=False)
    .rename_axis("litho_raw")
    .reset_index(name="count")
)
other_terms["percent_of_other"] = 100 * other_terms["count"] / len(other_audit)

print("\nTop raw descriptions classified as 'other':")
print(other_terms.head(30).to_string(index=False))

unknown_audit.to_csv(OUT / "phase1_unknown_macrostrat_records.csv", index=False)
other_audit.to_csv(OUT / "phase1_other_macrostrat_records.csv", index=False)
other_terms.to_csv(OUT / "phase1_other_raw_term_frequencies.csv", index=False)


## 4. Build the final model-set audit table

This joins the final covariates to the original Macrostrat response and documentation-depth information.

No row is reclassified here. The purpose is traceability.


In [ ]:
macro_join = macro.rename(columns={
    "latitude": "macro_latitude",
    "longitude": "macro_longitude",
    "litho_class": "macro_litho_class",
    "litho_raw": "macro_litho_raw",
})

audit = (
    cov
    .merge(
        macro_join[
            ["Mindat_id", "macro_latitude", "macro_longitude",
             "macro_litho_raw", "macro_litho_class", "api_status"]
        ],
        on="Mindat_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        completeness,
        on="Mindat_id",
        how="left",
        validate="one_to_one",
    )
)

audit["age_status"] = np.where(
    audit["age_category"].fillna("Unknown").astype(str).str.strip().eq("Unknown"),
    "unknown",
    "resolved",
)

# Confirm that the saved covariate class matches the class in the Macrostrat file.
mismatch = (
    audit["litho_class"].astype(str).str.lower()
    != audit["macro_litho_class"].astype(str).str.lower()
)
print("Lithology-class mismatches after joining:", int(mismatch.sum()))

audit.to_csv(OUT / "phase1_model_set_audit_table.csv", index=False)
print("Saved:", OUT / "phase1_model_set_audit_table.csv")
print("Shape:", audit.shape)


## 5. Missingness and status by deposit type


In [ ]:
def count_percent_crosstab(data, row, col):
    counts = pd.crosstab(data[row], data[col], dropna=False)
    pct = pd.crosstab(data[row], data[col], normalize="index", dropna=False) * 100
    return counts, pct

lith_type_counts, lith_type_pct = count_percent_crosstab(
    audit, "Deposit_type", "lithology_status"
)
age_type_counts, age_type_pct = count_percent_crosstab(
    audit, "Deposit_type", "age_status"
)

print("Lithology status by deposit type — counts")
print(lith_type_counts.to_string())
print("\nLithology status by deposit type — row percentages")
print(lith_type_pct.round(1).to_string())

print("\nAge status by deposit type — counts")
print(age_type_counts.to_string())
print("\nAge status by deposit type — row percentages")
print(age_type_pct.round(1).to_string())

lith_type_counts.to_csv(OUT / "phase1_lithology_status_by_deposit_type_counts.csv")
lith_type_pct.to_csv(OUT / "phase1_lithology_status_by_deposit_type_percent.csv")
age_type_counts.to_csv(OUT / "phase1_age_status_by_deposit_type_counts.csv")
age_type_pct.to_csv(OUT / "phase1_age_status_by_deposit_type_percent.csv")


## 6. Documentation depth by deposit type

The present counts are numbers of recorded mineral species within mineral families, not mineral abundances. These summaries provide the first documentation-bias diagnostic.


In [ ]:
richness_by_type = (
    audit.groupby("Deposit_type")
    .agg(
        n_deposits=("Mindat_id", "count"),
        median_recorded_species=("n_minerals_total", "median"),
        mean_recorded_species=("n_minerals_total", "mean"),
        q25_recorded_species=("n_minerals_total", lambda x: x.quantile(0.25)),
        q75_recorded_species=("n_minerals_total", lambda x: x.quantile(0.75)),
        median_mapped_families=("n_families_copper", "median"),
        mean_mapped_families=("n_families_copper", "mean"),
    )
    .reset_index()
)

print(richness_by_type.round(2).to_string(index=False))
richness_by_type.to_csv(OUT / "phase1_richness_by_deposit_type.csv", index=False)


## 7. Approximate continent summaries used by the original notebook

This reproduces the original coordinate-rule assignment only for audit consistency.

Before final submission, continent assignment should be checked against a proper geographic boundary dataset because the original rules are approximate.


In [ ]:
def assign_continent_original(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return "Unknown"
    if lat > 15 and -170 < lon < -50:
        return "North_America"
    if lat <= 15 and -120 < lon < -30:
        return "South_America"
    if lat > 35 and -25 < lon < 60:
        return "Europe"
    if lat > 0 and 60 < lon < 180:
        return "Asia"
    if lat <= 0 and 95 < lon < 180:
        return "Oceania"
    if -40 < lat < 40 and -25 < lon < 55:
        return "Africa"
    if lat <= -10 and 110 < lon < 180:
        return "Oceania"
    return "Other"

audit["continent_approx_original"] = audit.apply(
    lambda r: assign_continent_original(r["Latitude"], r["Longitude"]),
    axis=1,
)

lith_cont_counts, lith_cont_pct = count_percent_crosstab(
    audit, "continent_approx_original", "lithology_status"
)
age_cont_counts, age_cont_pct = count_percent_crosstab(
    audit, "continent_approx_original", "age_status"
)

richness_by_continent = (
    audit.groupby("continent_approx_original")
    .agg(
        n_deposits=("Mindat_id", "count"),
        median_recorded_species=("n_minerals_total", "median"),
        mean_recorded_species=("n_minerals_total", "mean"),
        median_mapped_families=("n_families_copper", "median"),
        mean_mapped_families=("n_families_copper", "mean"),
    )
    .reset_index()
)

print("Lithology status by approximate continent — counts")
print(lith_cont_counts.to_string())
print("\nLithology status by approximate continent — percentages")
print(lith_cont_pct.round(1).to_string())

lith_cont_counts.to_csv(OUT / "phase1_lithology_status_by_continent_counts.csv")
lith_cont_pct.to_csv(OUT / "phase1_lithology_status_by_continent_percent.csv")
age_cont_counts.to_csv(OUT / "phase1_age_status_by_continent_counts.csv")
age_cont_pct.to_csv(OUT / "phase1_age_status_by_continent_percent.csv")
richness_by_continent.to_csv(OUT / "phase1_richness_by_continent.csv", index=False)

# Save the audit table again with the reproduced continent field.
audit.to_csv(OUT / "phase1_model_set_audit_table.csv", index=False)


## 8. Export the strict and non-unknown subsets

These files make the previous subset-definition error explicit.

- `strict_resolved_lithology_ids.csv` excludes both `other` and `unknown`.
- `non_unknown_including_other_ids.csv` excludes only `unknown`.


In [ ]:
strict_ids = audit.loc[
    audit["lithology_status"] == "resolved",
    ["Mindat_id", "litho_class", "age_category", "Deposit_type"]
].copy()

non_unknown_ids = audit.loc[
    audit["lithology_status"] != "unknown",
    ["Mindat_id", "litho_class", "age_category", "Deposit_type"]
].copy()

strict_resolved_age_ids = audit.loc[
    (audit["lithology_status"] == "resolved") &
    (audit["age_status"] == "resolved"),
    ["Mindat_id", "litho_class", "age_category", "Deposit_type"]
].copy()

print("Strict resolved lithology:", len(strict_ids))
print("Non-unknown including 'other':", len(non_unknown_ids))
print("Strict resolved lithology and resolved age:", len(strict_resolved_age_ids))

strict_ids.to_csv(OUT / "strict_resolved_lithology_ids.csv", index=False)
non_unknown_ids.to_csv(OUT / "non_unknown_including_other_ids.csv", index=False)
strict_resolved_age_ids.to_csv(
    OUT / "strict_resolved_lithology_and_age_ids.csv",
    index=False
)


## 9. Create one audit archive


In [ ]:
import shutil

archive_base = WORK / "HGCTM_Phase1A_Audit_Outputs"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=OUT
)

print("Audit files:")
for p in sorted(OUT.iterdir()):
    print(" -", p.name)

print("\nDownload this archive from Kaggle Output:")
print(archive_path)
